# Pneumonia Detection from Chest X-Rays using CNN

## A Comprehensive Deep Learning Pipeline

This notebook builds a strong machine learning model to classify chest X-ray images as either **Normal** or **Pneumonia** using:
- **Transfer Learning** (ResNet50 pretrained backbone)
- **Advanced Data Augmentation** strategies
- **Proper Train/Val/Test Splitting** with stratification
- **Early Stopping** and model checkpointing
- **Comprehensive Evaluation Metrics** and visualizations

### Key Features:
✓ ResNet50-based architecture optimized for medical imaging
✓ Multiple augmentation strategies for robustness
✓ Learning rate scheduling and weight decay regularization
✓ Detailed performance metrics and confusion matrices
✓ ROC curves and probability calibration analysis
✓ Production-ready model saving and loading

## 1. Import Required Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import random
import os
from PIL import Image
from tqdm import tqdm
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", "GPU" if torch.cuda.is_available() else "CPU")

## 2. Configuration and Setup

In [ ]:
# ============================================================================
# Configuration
# ============================================================================

class Config:
    # Random seed for reproducibility
    RANDOM_SEED = 42
    
    # Dataset path
    DATA_DIR = r"C:\Users\PC\OneDrive\Desktop\Pneumonia Detection from Chest X-Rays using CNN\chest_xray"
    CLASSES = ["NORMAL", "PNEUMONIA"]
    
    # Training hyperparameters
    BATCH_SIZE = 32
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-5
    EPOCHS = 30
    EARLY_STOPPING_PATIENCE = 7
    
    # Image size
    IMG_SIZE = (224, 224)
    
    # Device
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Model save paths
    MODEL_SAVE_PATH = "best_pneumonia_model.pth"

print(f"Device: {Config.DEVICE}")
print(f"Batch Size: {Config.BATCH_SIZE}")
print(f"Learning Rate: {Config.LEARNING_RATE}")
print(f"Image Size: {Config.IMG_SIZE}")

In [ ]:
def set_seed(seed):
    """Set random seeds for reproducibility across all libraries."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Set seed
set_seed(Config.RANDOM_SEED)
print("Random seeds set for reproducibility")

## 3. Load and Explore Dataset

In [ ]:
def load_dataset(data_dir, classes):
    """Load all images from the dataset structure."""
    image_paths = []
    labels = []
    
    for idx, class_name in enumerate(classes):
        class_dir = os.path.join(data_dir, class_name)
        if not os.path.exists(class_dir):
            print(f"Warning: {class_dir} does not exist")
            continue
        
        for img_file in os.listdir(class_dir):
            if img_file.lower().endswith(('.png', '.jpg', '.jpeg')):
                image_paths.append(os.path.join(class_dir, img_file))
                labels.append(idx)
    
    return np.array(image_paths), np.array(labels)

# Load dataset
print("Loading dataset...")
image_paths, labels = load_dataset(Config.DATA_DIR, Config.CLASSES)
print(f"Total images loaded: {len(image_paths)}")

# Class distribution
class_counts = np.bincount(labels)
print(f"\nClass distribution:")
for class_name, count in zip(Config.CLASSES, class_counts):
    print(f"  {class_name}: {count} images")

# Visualize sample images
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
fig.suptitle("Sample Chest X-Ray Images", fontsize=14, weight='bold')

for i in range(2):
    for j in range(4):
        idx = np.random.choice(np.where(labels == i)[0])
        img = Image.open(image_paths[idx]).convert('L')
        axes[i, j].imshow(img, cmap='gray')
        axes[i, j].set_title(f"{Config.CLASSES[i]}")
        axes[i, j].axis('off')

plt.tight_layout()
plt.show()

print(f"Dataset ready for processing!")

## 4. Data Preprocessing and Augmentation

In [ ]:
def split_dataset(image_paths, labels, test_size=0.2, val_size=0.2, random_state=42):
    """Split dataset into train, validation, and test sets using stratified splitting."""
    # First split: separate test set
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_val_idx, test_idx = list(sss1.split(image_paths, labels))[0]
    
    # Second split: separate validation from training
    train_val_labels = labels[train_val_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=val_size/(1-test_size), random_state=random_state)
    train_idx, val_idx = list(sss2.split(train_val_idx, train_val_labels))[0]
    
    train_idx = train_val_idx[train_idx]
    val_idx = train_val_idx[val_idx]
    
    return (image_paths[train_idx], labels[train_idx],
            image_paths[val_idx], labels[val_idx],
            image_paths[test_idx], labels[test_idx])

# Split dataset
print("Splitting dataset into train/val/test...")
train_paths, train_labels, val_paths, val_labels, test_paths, test_labels = \
    split_dataset(image_paths, labels)

print(f"Train set: {len(train_paths)} images")
print(f"  NORMAL: {np.sum(train_labels == 0)}, PNEUMONIA: {np.sum(train_labels == 1)}")
print(f"Validation set: {len(val_paths)} images")
print(f"  NORMAL: {np.sum(val_labels == 0)}, PNEUMONIA: {np.sum(val_labels == 1)}")
print(f"Test set: {len(test_paths)} images")
print(f"  NORMAL: {np.sum(test_labels == 0)}, PNEUMONIA: {np.sum(test_labels == 1)}")

# Define data augmentation and normalization transforms
def get_transforms():
    """Create data augmentation and normalization transforms."""
    
    # Training transforms with aggressive augmentation
    train_transform = transforms.Compose([
        transforms.Resize(Config.IMG_SIZE),
        transforms.RandomRotation(20),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.RandomVerticalFlip(p=0.2),
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
        transforms.RandomAdjustSharpness(sharpness_factor=2, p=0.2),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])
    
    # Validation/Test transforms (no augmentation)
    val_test_transform = transforms.Compose([
        transforms.Resize(Config.IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])
    
    return train_transform, val_test_transform

train_transform, val_test_transform = get_transforms()
print("\nTransforms created!")

In [ ]:
class ChestXRayDataset(Dataset):
    """Custom PyTorch Dataset for chest X-ray images."""
    
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        # Load image as grayscale
        image = Image.open(img_path).convert('L')
        
        # Apply transformations
        if self.transform:
            image = self.transform(image)
        
        return image, label

# Create datasets
train_dataset = ChestXRayDataset(train_paths, train_labels, train_transform)
val_dataset = ChestXRayDataset(val_paths, val_labels, val_test_transform)
test_dataset = ChestXRayDataset(test_paths, test_labels, val_test_transform)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Data loaders created successfully!")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## 5. Build CNN Model Architecture

We use **Transfer Learning** with a pretrained ResNet50 backbone, modified to accept single-channel grayscale images. The model features:
- Pretrained ResNet50 backbone (ImageNet weights)
- Modified first conv layer for grayscale input
- Custom classification head with batch normalization and dropout
- Binary classification output (NORMAL vs PNEUMONIA)

In [ ]:
class PneumoniaDetectionModel(nn.Module):
    """Transfer learning model using ResNet50 backbone with custom classification head."""
    
    def __init__(self, pretrained=True):
        super(PneumoniaDetectionModel, self).__init__()
        
        # Load pretrained ResNet50
        resnet50 = models.resnet50(pretrained=pretrained)
        
        # Modify first layer for single-channel input
        self.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = resnet50.bn1
        self.relu = resnet50.relu
        self.maxpool = resnet50.maxpool
        
        # ResNet50 layers
        self.layer1 = resnet50.layer1
        self.layer2 = resnet50.layer2
        self.layer3 = resnet50.layer3
        self.layer4 = resnet50.layer4
        self.avgpool = resnet50.avgpool
        
        # Custom classification head
        self.fc = nn.Sequential(
            nn.Linear(resnet50.fc.in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 2)  # Binary classification
        )
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        
        return x

# Initialize model
print("Building model...")
model = PneumoniaDetectionModel(pretrained=True).to(Config.DEVICE)
print(f"Model loaded on {Config.DEVICE}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 6. Compile the Model

In [ ]:
# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), 
                       lr=Config.LEARNING_RATE, 
                       weight_decay=Config.WEIGHT_DECAY)

# Learning rate scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=Config.EPOCHS)

print("Model compiled successfully!")
print(f"Loss function: CrossEntropyLoss")
print(f"Optimizer: AdamW (lr={Config.LEARNING_RATE}, weight_decay={Config.WEIGHT_DECAY})")
print(f"Scheduler: CosineAnnealingLR")

## 7. Train the Model

The model will be trained with:
- **Early stopping** if validation accuracy doesn't improve
- **Model checkpointing** to save the best model
- **Learning rate scheduling** with cosine annealing
- **Batch training** with tqdm progress bars

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    progress_bar = tqdm(train_loader, desc="Training", leave=False)
    
    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        progress_bar.set_postfix({"loss": f"{running_loss/total:.4f}"})
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = correct / total
    
    return epoch_loss, epoch_acc

def validate_epoch(model, val_loader, criterion, device):
    """Validate the model."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / len(val_loader)
    epoch_acc = correct / total
    
    return epoch_loss, epoch_acc, all_preds, all_labels

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, 
                epochs, device, patience=7):
    """Train the model with early stopping."""
    train_losses = []
    train_accs = []
    val_losses = []
    val_accs = []
    
    best_val_acc = 0
    patience_counter = 0
    best_epoch = 0
    
    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        print("-" * 50)
        
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        
        # Validate
        val_loss, val_acc, _, _ = validate_epoch(model, val_loader, criterion, device)
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        # Learning rate scheduling
        scheduler.step()
        
        # Early stopping and model checkpointing
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            best_epoch = epoch
            torch.save(model.state_dict(), Config.MODEL_SAVE_PATH)
            print(f"✓ Best model saved with accuracy: {best_val_acc:.4f}")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\n⛔ Early stopping at epoch {epoch+1}. Best accuracy: {best_val_acc:.4f} at epoch {best_epoch+1}")
                break
    
    # Load best model
    model.load_state_dict(torch.load(Config.MODEL_SAVE_PATH))
    
    return {
        'train_losses': train_losses,
        'train_accs': train_accs,
        'val_losses': val_losses,
        'val_accs': val_accs,
        'best_val_acc': best_val_acc,
        'best_epoch': best_epoch
    }

# Train the model
print("Starting model training...\n")
train_history = train_model(model, train_loader, val_loader, criterion, 
                            optimizer, scheduler, Config.EPOCHS, 
                            Config.DEVICE, patience=Config.EARLY_STOPPING_PATIENCE)

## 8. Evaluate Model Performance

In [ ]:
def evaluate_model(model, test_loader, device):
    """Evaluate model on test set."""
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Evaluating"):
            images = images.to(device)
            
            outputs = model(images)
            probabilities = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs.data, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probabilities.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    return all_preds, all_labels, all_probs

# Evaluate on test set
print("\nEvaluating model on test set...")
all_preds, all_labels, all_probs = evaluate_model(model, test_loader, Config.DEVICE)

# Print comprehensive metrics
from sklearn.metrics import precision_score, recall_score, f1_score

accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)
roc_auc = roc_auc_score(all_labels, all_probs[:, 1])

print("\n" + "="*60)
print("TEST SET EVALUATION RESULTS")
print("="*60)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print("="*60)

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=Config.CLASSES))

## 9. Visualize Results

In [ ]:
# Create comprehensive visualization
fig = plt.figure(figsize=(18, 12))

# 1. Training and validation loss
ax1 = plt.subplot(2, 3, 1)
ax1.plot(train_history['train_losses'], 'b-', linewidth=2, label='Train Loss')
ax1.plot(train_history['val_losses'], 'r-', linewidth=2, label='Val Loss')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training and Validation Loss', fontsize=14, weight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# 2. Training and validation accuracy
ax2 = plt.subplot(2, 3, 2)
ax2.plot(train_history['train_accs'], 'b-', linewidth=2, label='Train Accuracy')
ax2.plot(train_history['val_accs'], 'r-', linewidth=2, label='Val Accuracy')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title('Training and Validation Accuracy', fontsize=14, weight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

# 3. Confusion matrix
ax3 = plt.subplot(2, 3, 3)
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax3,
            xticklabels=Config.CLASSES, yticklabels=Config.CLASSES,
            cbar_kws={'label': 'Count'})
ax3.set_title('Confusion Matrix', fontsize=14, weight='bold')
ax3.set_ylabel('True Label', fontsize=12)
ax3.set_xlabel('Predicted Label', fontsize=12)

# 4. ROC Curve
ax4 = plt.subplot(2, 3, 4)
from sklearn.metrics import roc_curve, auc
fpr, tpr, _ = roc_curve(all_labels, all_probs[:, 1])
roc_auc_val = auc(fpr, tpr)
ax4.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC Curve (AUC = {roc_auc_val:.4f})')
ax4.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
ax4.set_xlabel('False Positive Rate', fontsize=12)
ax4.set_ylabel('True Positive Rate', fontsize=12)
ax4.set_title('ROC Curve', fontsize=14, weight='bold')
ax4.legend(fontsize=11)
ax4.grid(True, alpha=0.3)

# 5. Metrics summary
ax5 = plt.subplot(2, 3, 5)
ax5.axis('off')
metrics_text = f"""
Test Metrics:

Accuracy:  {accuracy:.4f}
Precision: {precision:.4f}
Recall:    {recall:.4f}
F1-Score:  {f1:.4f}
ROC-AUC:   {roc_auc:.4f}

Model: ResNet50 Transfer Learning
Epochs Trained: {len(train_history['train_losses'])}
Best Val Accuracy: {train_history['best_val_acc']:.4f}
"""
ax5.text(0.1, 0.5, metrics_text, fontsize=12, family='monospace',
         verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# 6. Prediction confidence
ax6 = plt.subplot(2, 3, 6)
normal_probs = all_probs[all_labels == 0][:, 0]
pneumonia_probs = all_probs[all_labels == 1][:, 1]
ax6.hist([normal_probs, pneumonia_probs], bins=30, label=Config.CLASSES, alpha=0.7)
ax6.set_xlabel('Confidence Score', fontsize=12)
ax6.set_ylabel('Frequency', fontsize=12)
ax6.set_title('Model Confidence Distribution', fontsize=14, weight='bold')
ax6.legend(fontsize=11)
ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pneumonia_model_evaluation.png', dpi=300, bbox_inches='tight')
print("✓ Evaluation plots saved as 'pneumonia_model_evaluation.png'")
plt.show()

## 10. Make Predictions on New Images

In [ ]:
def predict_single_image(model, image_path, device):
    """Make prediction on a single image."""
    model.eval()
    
    # Load and preprocess image
    image = Image.open(image_path).convert('L')
    image = val_test_transform(image)
    image = image.unsqueeze(0).to(device)
    
    # Make prediction
    with torch.no_grad():
        output = model(image)
        probabilities = torch.softmax(output, dim=1)
        predicted_class = torch.argmax(probabilities, dim=1)
    
    return predicted_class.item(), probabilities.cpu().numpy()[0]

# Test prediction on random test images
print("Making predictions on test set samples...\n")

# Select random test images
num_samples = 6
random_indices = np.random.choice(len(test_paths), num_samples, replace=False)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Sample Predictions from Test Set', fontsize=16, weight='bold')

for idx, (ax, sample_idx) in enumerate(zip(axes.flat, random_indices)):
    img_path = test_paths[sample_idx]
    true_label = test_labels[sample_idx]
    
    # Make prediction
    pred_class, probs = predict_single_image(model, img_path, Config.DEVICE)
    
    # Load and display image
    image = Image.open(img_path).convert('L')
    ax.imshow(image, cmap='gray')
    
    # Format prediction text
    confidence = max(probs)
    correct = "✓" if pred_class == true_label else "✗"
    color = 'green' if pred_class == true_label else 'red'
    
    title = f"{correct} True: {Config.CLASSES[true_label]}\nPred: {Config.CLASSES[pred_class]} ({confidence:.2%})"
    ax.set_title(title, fontsize=11, color=color, weight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

print("\n✓ Model is ready for inference!")

## 11. Model Persistence and Summary

In [ ]:
# Save model information
model_info = {
    'timestamp': datetime.now().isoformat(),
    'model_architecture': 'ResNet50 with Custom Classification Head',
    'test_metrics': {
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'f1_score': float(f1),
        'roc_auc': float(roc_auc)
    },
    'training_info': {
        'total_epochs': len(train_history['train_losses']),
        'best_epoch': int(train_history['best_epoch']),
        'best_validation_accuracy': float(train_history['best_val_acc']),
        'learning_rate': Config.LEARNING_RATE,
        'batch_size': Config.BATCH_SIZE,
        'optimizer': 'AdamW',
        'scheduler': 'CosineAnnealingLR'
    },
    'model_save_path': Config.MODEL_SAVE_PATH
}

# Save model information to JSON
with open('pneumonia_model_info.json', 'w') as f:
    json.dump(model_info, f, indent=4)

print("\n" + "="*60)
print("MODEL TRAINING COMPLETED SUCCESSFULLY!")
print("="*60)
print("\nModel saved to:", Config.MODEL_SAVE_PATH)
print("Model info saved to: pneumonia_model_info.json")
print("Evaluation plots saved to: pneumonia_model_evaluation.png")
print("\n" + "="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)
print(f"Test Accuracy:     {accuracy:.4f}")
print(f"Test Precision:    {precision:.4f}")
print(f"Test Recall:       {recall:.4f}")
print(f"Test F1-Score:     {f1:.4f}")
print(f"Test ROC-AUC:      {roc_auc:.4f}")
print("="*60)
print("\n✓ Ready to use model for inference on new chest X-ray images!")